Group-level aggregation FP benchmark on unseen (LOBO held-out) HC batches. 2 baselines (naive stouffer, same-batch-split extreme_mwu) vs 3 improved variants (covariate-matched pseudo-HC extreme_mwu, per-sample recentered stouffer, bootstrap empirical-null stouffer). Table output only, no plots. Swap/add methods in `AGG_METHODS`.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats import mannwhitneyu, norm
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.Benchmark import db_hit_compare as dc
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.core.marginal_rqr import marginal_nb_rqr
from MixedEffectsModeling.core.shash import shash_transform_to_z
from MixedEffectsModeling.validation.lobo_engine import load_full_data
from MixedEffectsModeling.validation.ppc_simulate import simulate_marginal_nb

PERTURB_DIR = config.ROOT / "MixedEffectsModeling" / "Perturbation_Results"
PERTURB_DIR.mkdir(exist_ok=True)

N_PERTURB_GENES = 500
LOG2FCS = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
N_BOOTSTRAP = 5
ALPHA = 0.05
Z_THRESH = 2.0  # extreme_mwu deviation cutoff
K_MATCH = 10    # covariate-matched pseudo-HC: nearest neighbors per group-A sample

In [ ]:
data = load_full_data()

_meta_rows = []
for bdir in sorted(config.LOBO_MIXED_DIR.iterdir()):
    meta_path = bdir / "meta.json"
    if not meta_path.exists():
        continue
    meta = json.loads(meta_path.read_text())
    n_hc_test = int(sum(meta["test_is_hc"]))
    _meta_rows.append(dict(batch_id=meta["batch_id"], safe_dir=bdir.name, n_hc_test=n_hc_test))
lobo_batches = pd.DataFrame(_meta_rows).sort_values("n_hc_test", ascending=False).reset_index(drop=True)

# unseen-batch HC split needs enough samples per half -- n_hc_test>=25 keeps each half >=12
BATCHES = lobo_batches[lobo_batches.n_hc_test >= 25]["batch_id"].tolist()
print(lobo_batches[lobo_batches.n_hc_test >= 25])

In [ ]:
_cache = {}
name2row = {n: i for i, n in enumerate(data["names"])}

# global in-sample HC pool for covariate-matched pseudo-HC (production's own reference set,
# see db_hit_compare.py extreme_mwu_group_z/wolfers_chi2_group_z)
hc_meta_global = pd.read_csv(dc.ZDIR / "hc_meta.csv")
Z_hc_global = np.load(dc.ZDIR / "Z_hc_shash.npy")
gene_names_global = pickle.load(open(dc.ZDIR / "gene_names.pkl", "rb"))
gene_pos_global = {g: j for j, g in enumerate(gene_names_global)}
hc_global_rows = np.array([name2row[s] for s in hc_meta_global["sample"]])


def load_batch(batch_id):
    if batch_id in _cache:
        return _cache[batch_id]
    safe = lobo_batches.set_index("batch_id").loc[batch_id, "safe_dir"]
    bdir = config.LOBO_MIXED_DIR / safe
    meta = json.loads((bdir / "meta.json").read_text())
    gene_names = pickle.load(open(bdir / "gene_names.pkl", "rb"))
    Z_test = np.load(bdir / "Z_test_shash.npy")
    test_is_hc = np.array(meta["test_is_hc"])

    fits = pd.read_csv(bdir / "model_fits.csv").set_index("gene")
    fits = fits[fits["ok"]]
    shash_p = pd.read_csv(bdir / "shash_params.csv").set_index("gene")
    universe = [g for g in fits.index if g in shash_p.index and g in gene_names]  # model-route only

    is_hc, batch, small = data["is_hc"], data["batch"], data["small_hc_batches"]
    tr_idx = np.where(is_hc & (batch != batch_id) & ~np.isin(batch, list(small)))[0]
    scaler = StandardScaler().fit(data["X_raw"][tr_idx])

    hc_test_names = np.array(meta["test_names"])[test_is_hc]
    hc_test_rows = np.array([name2row[n] for n in hc_test_names])

    # covariate-matching candidate pool: global in-sample HC minus this batch's own HC (no leakage)
    cand_mask = (hc_meta_global["batch"] != batch_id).values
    cand_rows = hc_global_rows[cand_mask]
    cand_std = scaler.transform(data["X_raw"][cand_rows])
    hc_col_idx = np.array([gene_pos_global[g] for g in gene_names])
    cand_Z = Z_hc_global[np.ix_(np.where(cand_mask)[0], hc_col_idx)]

    out = dict(gene_names=gene_names, gene_pos={g: j for j, g in enumerate(gene_names)},
               Z_hc=Z_test[test_is_hc], universe=universe, fits=fits, shash_p=shash_p,
               scaler=scaler, hc_test_rows=hc_test_rows, cand_std=cand_std, cand_Z=cand_Z)
    _cache[batch_id] = out
    return out


def matched_pseudo_hc(b, rows_A, k=K_MATCH):
    X_A = b["scaler"].transform(data["X_raw"][rows_A])
    d = cdist(X_A, b["cand_std"])
    nn = np.argsort(d, axis=1)[:, :k]
    matched = np.unique(nn.ravel())
    return b["cand_Z"][matched]


def mu_alpha_tau2(b, genes, sample_rows):
    Xs = b["scaler"].transform(data["X_raw"][sample_rows])
    Xa = np.column_stack([np.ones(len(sample_rows)), Xs])
    mu, alpha, tau2 = {}, {}, {}
    fits = b["fits"]
    mu_cols = [c for c in fits.columns if c.startswith("mu_coef_")]
    disp_cols = [c for c in fits.columns if c.startswith("disp_coef_")]
    for g in genes:
        row = fits.loc[g]
        mu_coef = row[mu_cols].values.astype(float)
        disp_coef = row[disp_cols].values.astype(float)
        mu[g] = np.clip(np.exp(Xa @ np.nan_to_num(mu_coef, nan=0.0)), 1e-6, 1e8)
        alpha[g] = (np.exp(-Xa @ np.nan_to_num(disp_coef, nan=0.0)) if not np.all(np.isnan(disp_coef))
                    else np.full(len(sample_rows), float(row["trend_alpha"])))
        tau2[g] = float(row["tau2"])
    return mu, alpha, tau2


def perturb_genes(b, genes, sample_rows, log2fc, seed):
    mu, alpha, tau2 = mu_alpha_tau2(b, genes, sample_rows)
    shash_p = b["shash_p"]
    z_of = {}
    for g in genes:
        mu_shift = mu[g] * (2 ** log2fc)
        y_pert = simulate_marginal_nb(mu_shift, alpha[g], tau2[g], n_reps=1, seed=seed + hash(g) % 9973)[0]
        z_raw = marginal_nb_rqr(y_pert, mu[g], alpha[g], tau2[g], seed=seed + hash(g) % 9973 + 1)
        srow = shash_p.loc[g]
        z = shash_transform_to_z(z_raw, srow.xi, srow.eta, srow.eps, srow.delta) if srow.ok else z_raw
        z_of[g] = np.clip(z, -50, 50)  # shash tail can blow up in float32 for extreme injected effects
    return z_of

In [ ]:
def agg_stouffer(Z_A, refs):
    finite = np.isfinite(Z_A)
    n = finite.sum(axis=0)
    s = np.where(finite, Z_A, 0.0).sum(axis=0)
    return np.divide(s, np.sqrt(n), out=np.full(n.shape, np.nan), where=n > 0)


def _mwu_vs_ref(Z_A, Z_ref, z_thresh=Z_THRESH):
    ind_A = np.where(np.isfinite(Z_A), (np.abs(Z_A) > z_thresh).astype(float), np.nan)
    ind_ref = np.where(np.isfinite(Z_ref), (np.abs(Z_ref) > z_thresh).astype(float), np.nan)
    res = mannwhitneyu(ind_A, ind_ref, axis=0, nan_policy="omit", alternative="two-sided")
    sign = np.sign(np.nanmean(ind_A, axis=0) - np.nanmean(ind_ref, axis=0))
    return sign * norm.isf(res.pvalue / 2)


def agg_extreme_mwu(Z_A, refs):
    # baseline: same-batch split HC as reference (needs a locally paired HC group)
    return _mwu_vs_ref(Z_A, refs["same_batch"])


def agg_extreme_mwu_covmatch(Z_A, refs):
    # improved: covariate-matched pseudo-HC drawn from the global in-sample pool -- no
    # local paired HC required, works for a genuinely new/unseen batch
    return _mwu_vs_ref(Z_A, refs["matched"])


def agg_stouffer_recentered(Z_A, refs):
    # improved: subtract each sample's own median Z (over the full universe) before summing,
    # to remove a shared per-sample batch shift that naive Stouffer cannot distinguish from signal
    centered = Z_A - np.nanmedian(Z_A, axis=1, keepdims=True)
    return agg_stouffer(centered, refs)


def agg_stouffer_permnull(Z_A, refs, n_boot=200, seed=0):
    # improved: empirical bootstrap SE of the group mean (over samples) instead of assuming
    # unit per-sample variance -- captures inflated variance from within-batch correlation.
    # W (n_boot x n) resample-count weights avoids materializing an (n_boot, n, n_genes) array.
    rng = np.random.default_rng(seed)
    n = Z_A.shape[0]
    Z_filled = np.where(np.isfinite(Z_A), Z_A, 0.0)
    idx = rng.integers(0, n, size=(n_boot, n))
    W = np.stack([np.bincount(row, minlength=n) for row in idx])
    boot_mean = (W @ Z_filled) / n
    obs_mean = np.nanmean(Z_A, axis=0)
    se = boot_mean.std(axis=0, ddof=1)
    return np.divide(obs_mean, se, out=np.full(obs_mean.shape, np.nan), where=se > 0)


# add a new aggregation method here, then delete Perturbation_Results/sweep.csv to rerun it
AGG_METHODS = {
    "stouffer": agg_stouffer,
    "extreme_mwu": agg_extreme_mwu,
    "extreme_mwu_covmatch": agg_extreme_mwu_covmatch,
    "stouffer_recentered": agg_stouffer_recentered,
    "stouffer_permnull": agg_stouffer_permnull,
}

In [ ]:
def run_sweep():
    cache_path = PERTURB_DIR / "sweep.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)

    rows = []
    for batch_id in BATCHES:
        b = load_batch(batch_id)
        n_hc = len(b["hc_test_rows"])
        universe = b["universe"]

        for boot in range(N_BOOTSTRAP):
            rng = np.random.default_rng(1000 * boot + hash(batch_id) % 9973)
            perm = rng.permutation(n_hc)
            idx_A, idx_B = perm[: n_hc // 2], perm[n_hc // 2 :]
            rows_A = b["hc_test_rows"][idx_A]
            refs = dict(same_batch=b["Z_hc"][idx_B], matched=matched_pseudo_hc(b, rows_A))

            for log2fc in LOG2FCS:
                pert_genes = list(rng.choice(universe, min(N_PERTURB_GENES, len(universe)), replace=False))
                z_of = perturb_genes(b, pert_genes, rows_A, log2fc, seed=int(1e6 * log2fc) + boot)

                Z_A = b["Z_hc"][idx_A].copy()
                labels = np.zeros(len(b["gene_names"]), dtype=bool)
                for g in pert_genes:
                    j = b["gene_pos"][g]
                    Z_A[:, j] = z_of[g]
                    labels[j] = True

                for name, fn in AGG_METHODS.items():
                    gz = fn(Z_A, refs)
                    finite = np.isfinite(gz)
                    p = np.full(len(gz), np.nan)
                    p[finite] = 2 * norm.sf(np.abs(gz[finite]))
                    reject = np.zeros(len(gz), dtype=bool)
                    reject[finite] = bh_fdr_reject(p[finite], q=ALPHA)
                    tp, fp = int((reject & labels).sum()), int((reject & ~labels).sum())
                    fn_, tn = int((~reject & labels).sum()), int((~reject & ~labels).sum())
                    rows.append(dict(batch=batch_id, boot=boot, log2fc=log2fc, method=name,
                                     n_universe=len(gz), n_perturbed=len(pert_genes),
                                     tp=tp, fp=fp, fn=fn_, tn=tn))
        print(batch_id, "done", flush=True)

    df = pd.DataFrame(rows)
    df.to_csv(cache_path, index=False)
    return df


sweep = run_sweep()
sweep.head()

In [ ]:
counts = sweep.groupby(["method", "log2fc"])[["tp", "fp", "fn", "tn"]].sum()
tp, fp, fn, tn = counts["tp"], counts["fp"], counts["fn"], counts["tn"]
summary = pd.DataFrame(dict(
    TP=tp, FP=fp, FN=fn, TN=tn,
    Precision=tp / (tp + fp), Sensitivity=tp / (tp + fn), Specificity=tn / (tn + fp),
    Accuracy=(tp + tn) / (tp + fp + fn + tn), FDR=fp / (tp + fp),
    F1=2 * tp / (2 * tp + fp + fn),
)).round(3)
display(summary)

null_calib = sweep[sweep.log2fc == 0].groupby("method")["fp"].agg(["mean", "max"])
print("null (log2fc=0) FP calibration -- should be near 0 if independence assumption holds:")
display(null_calib)